# Inputs for cached concept analysis
Creates local analysis inputs from labels and IDs stored with the cached embeddings; no API or encoder is needed.

In [1]:
from pathlib import Path

# Resolve the project from either its root or a notebook's folder.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in [current_directory, *current_directory.parents]
    if (path / "requirements.txt").is_file() and (path / "data_preparation").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Keep the three project folders and requirements.txt together.")
DATA_DIR = PROJECT_ROOT / "data_preparation"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments"
METRICS_DIR = PROJECT_ROOT / "metrics_export"

import numpy as np
import pandas as pd

with np.load(EXPERIMENT_DIR / "representations_frozen_15.npz", allow_pickle=False) as cache:
    cached_rows = pd.DataFrame({
        "row_id": cache["original_row_ids"],
        "dataset": cache["original_datasets"],
        "split": cache["original_splits"],
        "label": cache["original_labels"]})
assert cached_rows.row_id.is_unique
cached_rows.to_csv(DATA_DIR / "split.csv", index=False)

# Reproduce the original fixed sample without source text, API calls or model inference.
random_pair_rng = np.random.default_rng(17)
prepared_pair_frames = []
for dataset in ["deepset", "safeguard"]:
    members = []
    for label in [1, 0]:
        sources = cached_rows[cached_rows["split"].eq("test")
            & cached_rows.dataset.eq(dataset) & cached_rows.label.eq(label)].sort_values("row_id")
        positions = random_pair_rng.choice(len(sources), size=10, replace=False)
        members.append(sources.iloc[positions].reset_index(drop=True))
    injections, benign = members
    prepared_pair_frames.append(pd.DataFrame({
        "pair_id": [f"{dataset}:random:{index:02d}" for index in range(10)],
        "dataset": dataset, "injection_row_id": injections.row_id, "benign_row_id": benign.row_id}))
prepared_pairs = pd.concat(prepared_pair_frames, ignore_index=True)
assert len(set(prepared_pairs.injection_row_id) | set(prepared_pairs.benign_row_id)) == 40
prepared_pairs.to_csv(DATA_DIR / "random_concept_pairs.csv", index=False)
print("Prepared cached-analysis inputs:", len(cached_rows), "source records and", len(prepared_pairs), "pairs")


Prepared cached-analysis inputs: 1640 source records and 20 pairs
